# scHiCAR linked multiomics framework

This notebook shows the recommended U-Chrom layout for scHiCAR data after spatial mapping to MERFISH or another tissue atlas.

The key design decision is axis separation:

- `ChromData` stores the spatialized U-Chrom view, usually on the MERFISH or tissue-cell axis.
- MuData stores raw and mapped single-cell modalities such as scHiCAR RNA, ATAC-derived features, contact-derived features, and MERFISH RNA.
- scool stores raw single-cell Hi-C contact matrices keyed by scHiCAR cell or RNA barcode.

Raw scHiCAR contact matrices are not xyz coordinates. If per-bin 3D coordinates were not directly measured or reconstructed, `coords` should remain explicitly missing.

In [ ]:
import numpy as np
import pandas as pd

from uchrom import ChromData, UChromProject

## 1. Spatialized ChromData layer

After Tangram or a similar mapper, the primary U-Chrom object can use the tissue or MERFISH cell axis. Each spot is a `cell x genome_bin` row.

The MERFISH centroid is cell-level spatial metadata. It is not a chromatin-bin xyz coordinate, so this example keeps `coords` as all `NaN` and records the reason in `uns['coordinate_status']`.

In [ ]:
cells = pd.DataFrame(
    {
        "cell_id": ["merfish_cell_001", "merfish_cell_002"],
        "x_um": [125.0, 142.5],
        "y_um": [80.0, 87.0],
        "subclass": ["L5 IT", "L6 CT"],
        "spatial_source": ["MERFISH centroid", "MERFISH centroid"],
    }
).set_index("cell_id")

bins = pd.DataFrame(
    {
        "chrom": ["chr1", "chr1", "chr1"],
        "start": [0, 1_000_000, 2_000_000],
        "end": [1_000_000, 2_000_000, 3_000_000],
    }
)

spots = []
for cell_id in cells.index:
    for _, row in bins.iterrows():
        spots.append(
            {
                "cell_id": cell_id,
                "trace_id": cell_id,
                "chrom": row.chrom,
                "start": row.start,
                "end": row.end,
            }
        )
spots = pd.DataFrame(spots)
coords = np.full((len(spots), 3), np.nan)

cd = ChromData(
    coords,
    spots,
    cells=cells,
    uns={
        "dataset": "scHiCAR-to-MERFISH demonstration",
        "coordinate_status": "missing_per_bin_3d_coordinates_all_coords_nan",
        "spatial_mapping": {
            "method": "Tangram map_cells_to_space",
            "spatial_coordinates": "cell-level inferred or atlas coordinates, not per-bin xyz",
        },
    },
)

cd.coord_status()

## 2. Link MuData and scool instead of embedding them

The `.h5cd` stores lightweight links and provenance only. Large matrix objects stay in their native containers:

- `.h5mu` for multi-modal cell matrices and mapping diagnostics.
- `.scool` for raw single-cell Hi-C contacts.

The paths below are examples. Replace them with local paths when running on real GSE305439/scHiCAR output.

In [ ]:
cd.link_mudata(
    "data/schicar/GSE305439_scHiCAR_MOp_MERFISH_linked.h5mu",
    key="schicar_mop",
    modalities=["schicar_rna", "schicar_atac", "hic_features", "merfish_spatial"],
    cell_axis="scHiCAR cell axis with explicit mapping to MERFISH cells",
    spatial_source="Tangram mapping to MERFISH atlas",
    not_measured_spatial=True,
)

cd.link_scool(
    "data/schicar/GSE305439_DNA_1Mb_MOp_RNAbarcode.scool",
    key="schicar_contacts",
    barcode_source="GSE305439 RNA barcode",
    genome_assembly="mm10",
    bin_size=1_000_000,
    coordinate_status="raw contacts, not xyz coordinates",
)

cd.describe_links()

## 3. Validate link metadata

`check_files=False` validates metadata shape and coordinate semantics without requiring the external `.h5mu` or `.scool` files to exist in this tutorial environment. Use `check_files=True` in a real analysis directory.

In [ ]:
issues = cd.validate_links(check_files=False)
issues

## 4. Project-level access

`UChromProject` is a thin convenience wrapper around `ChromData` plus linked external modalities. It does not force MuData or scool into the `.h5cd`; it keeps the biological axes explicit.

In [ ]:
project = UChromProject(cd)
project.describe()

## Practical interpretation

For scHiCAR mapped into tissue space, the organized data should be read as:

- `ChromData.cells`: MERFISH or tissue cells with spatial metadata.
- `ChromData.spots`: rows for `cell x genomic bin` features.
- `ChromData.coords`: missing unless per-bin 3D coordinates are directly measured or reconstructed.
- `linked_mudata`: raw and mapped multi-modal single-cell matrices.
- `linked_scool`: raw single-cell Hi-C contact matrices.

This keeps projected spatial evidence separate from raw 3D-genome contact evidence, which is required for defensible downstream interpretation.